In [ ]:
from rich import print
import pandas as pd
import requests
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from model import SessionLocal, Trade, Country, Product

session = SessionLocal()

In [ ]:
trade = session.query(Trade).all()
len(trade)

875

In [ ]:
continents = session.query(Country.continent_name).distinct().all()
continents = [x[0] for x in continents]
print("continents> ", continents)

continent_translation = {
    "Oceania": "as",
    "North America": "na",
    "South America": "sa",
    "Africa": "af",
    "Europe": "eu",
    "Asia": "as",
    "Middle-East": "as",
    'Antarctica': 'as'
}


result = set()
for i in session.query(Country).filter(Country.population >= 10_000_000).all():
    i = f"{continent_translation[i.continent_name]}{i.iso_3}".lower()
    result.add(i)

result = list(result)
countries = result
print(len(countries))

continents> 
['Oceania', 'North America', 'South America', 'Antarctica', 'Africa', 'Europe', 'Asia', 'Middle-East']

90

In [ ]:
desired = ["Cocoa Beans", "Honey", "Coffee", "Tea", "Rice", "Copper Ore"]
final_list = []
url = "https://api-v2.oec.world/tesseract/members?cube=trade_i_baci_a_22&level=HS4"
response = requests.get(url=url)
r = requests.get(url)
print(r.status_code, r.reason)
if r.status_code == 200:
    js = r.json()

    products = js["members"]
    for i in products:
        if i['caption'] in desired:
            final_list.append(i)
    
    for x in final_list:
        product = session.query(Product).filter(Product.id == x['key']).first()
        if not product:
            product = Product()

        product.id = x['key']
        product.name = x['caption']

        session.add(product)
    session.commit()


products = session.query(Product).all()
products = [x.id for x in products]
print(products)


200 OK

['52709', '52603', '52711', '10409', '20901', '20902', '21006', '41801']

In [ ]:
years = [2024, 2023]
for key in products[0:1]:
    for country in countries:
        for year in years:

            url = f"https://api-v2.oec.world/tesseract/data.jsonrecords?cube=trade_i_baci_a_17&drilldowns=HS4,Year,Exporter+Country, Importer+Country&measures=Trade+Value&include=Year:{year};Exporter+Country:{country};HS4:{key}&limit=500,0"
            r = requests.get(url)
            r.raise_for_status()
            js = r.json()
            annotations = js.get("annotations")
            page = js.get("page")
            data = js.get("data", [])
            df = pd.DataFrame(data)

            if len(df) == 0:
                continue

            print(key, country, year, len(df))
            df.columns = df.columns.str.lower().str.replace(' ', '_')

            for row in df.itertuples(index=False):
                product_id = int(row.hs4_id)
                year = int(row.year)
                exporter = row.exporter_country_id[2:].upper()
                importer = row.importer_country_id[2:].upper()
                value = int(row.trade_value)

                trade = (
                    session.query(Trade)
                    .filter(Trade.product_id == product_id)
                    .filter(Trade.exporter == exporter)
                    .filter(Trade.importer == importer)
                    .filter(Trade.year == year)
                    .first()
                )
                if not trade:
                    trade = Trade()

                trade.product_id=product_id
                trade.exporter=exporter
                trade.importer=importer
                trade.year=year
                trade.value = value

                session.add(trade)
            session.commit()

52709 assau
[2024, 2023]

52709 assau
[2024, 2023]

52709 askor
[2024, 2023]

52709 askor
[2024, 2023]

52709 aslka
[2024, 2023]

...

52709 eugbr
[2024, 2023]

52709 afmwi
[2024, 2023]

52709 afmwi
[2024, 2023]

52709 asbgd
[2024, 2023]

52709 asbgd
[2024, 2023]

In [ ]:
data = session.query(Trade).all()
len(data)


8144